# WES Analysis
Organoid whole-exome sequencing results: coverage depth, variant profiles,
allele frequency spectra, and comparison against scRNA cell-line runs.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

DB_PATH = 'results/variants.db'

def query(sql, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

# Load all runs; tag WES by run_id convention
all_runs = query("""
    SELECT r.run_id, s.name AS sample, s.cell_line,
           r.mapping_rate, r.total_raw, r.total_filt, r.run_date
    FROM runs r
    JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_id
""")
all_runs['data_type'] = all_runs['run_id'].apply(
    lambda x: 'WES' if '_wes_' in x.lower() else 'sc/bulk'
)
all_runs['label'] = all_runs['cell_line'].fillna(all_runs['sample'])

wes_runs = all_runs[all_runs['data_type'] == 'WES'].copy()
sc_runs  = all_runs[all_runs['data_type'] == 'sc/bulk'].copy()

print(f'Total runs: {len(all_runs)}  (WES: {len(wes_runs)}, sc/bulk: {len(sc_runs)})')
if wes_runs.empty:
    print('No WES runs in database yet.')
else:
    display(wes_runs[['run_id', 'sample', 'mapping_rate', 'total_filt', 'run_date']]
            .style.format({'mapping_rate': '{:.2f}', 'total_filt': '{:,.0f}'}))

## WES sample overview
Mapping rate and filtered variant counts for the organoid WES samples.

In [ ]:
if wes_runs.empty:
    print('No WES runs available.')
else:
    wes_sorted = wes_runs.sort_values('mapping_rate', ascending=True)
    map_colors = ['#e63946' if v < 95 else '#6d597a' for v in wes_sorted['mapping_rate']]
    wes_sorted2 = wes_runs.sort_values('total_filt', ascending=True)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Mapping rate (%) — WES', 'Filtered variants — WES'),
        horizontal_spacing=0.12
    )
    fig.add_trace(go.Bar(
        y=wes_sorted['sample'], x=wes_sorted['mapping_rate'],
        orientation='h', marker_color=map_colors,
        hovertemplate='%{y}<br>%{x:.2f}%<extra></extra>'
    ), row=1, col=1)
    fig.add_vline(x=95, line_dash='dash', line_color='#e63946',
                  annotation_text='95%', row=1, col=1)
    fig.add_trace(go.Bar(
        y=wes_sorted2['sample'], x=wes_sorted2['total_filt'],
        orientation='h', marker_color='#6d597a',
        hovertemplate='%{y}<br>%{x:,}<extra></extra>'
    ), row=1, col=2)
    fig.update_layout(
        height=max(350, 30 * len(wes_runs)),
        title=f'WES run overview — {len(wes_runs)} organoid samples',
        showlegend=False
    )
    fig.update_xaxes(range=[0, 102], row=1, col=1)
    fig.show()

## Per-sample depth and AF distributions
WES runs should show much higher depth than scRNA runs (~80–120x vs 10–30x),
and a clean bimodal AF distribution at 0.5 (het) and 1.0 (hom).

In [ ]:
if wes_runs.empty:
    print('No WES runs available.')
else:
    wes_select = widgets.Dropdown(
        options=list(zip(wes_runs['sample'], wes_runs['run_id'])),
        description='WES sample:',
        layout=widgets.Layout(width='55%')
    )
    out_wes = widgets.Output()

    def show_wes_depth(run_id):
        geno = query("""
            SELECT g.depth, g.allele_freq, v.chromosome AS chrom, g.quality AS qual
            FROM genotype_calls g
            JOIN variants v ON g.variant_id = v.variant_id
            WHERE g.run_id = ? AND g.filter_status = 'PASS'
            LIMIT 1000000
        """, (run_id,))
        if geno.empty:
            print(f'No passing variants for {run_id}.')
            return

        sample_name = wes_runs[wes_runs['run_id'] == run_id]['sample'].iloc[0]

        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Depth distribution (log scale)',
                'Allele frequency spectrum',
                'QUAL distribution',
                'Chromosomal distribution'
            ),
            horizontal_spacing=0.1, vertical_spacing=0.15
        )
        fig.add_trace(go.Histogram(
            x=np.log10(geno['depth'].clip(lower=1)), nbinsx=60,
            marker_color='#6d597a', name='log10(depth)'
        ), row=1, col=1)
        fig.update_xaxes(
            title_text='log10(depth)',
            tickvals=[1, 2, 3], ticktext=['10x', '100x', '1000x'],
            row=1, col=1
        )

        fig.add_trace(go.Histogram(
            x=geno['allele_freq'], nbinsx=50,
            marker_color='#e9c46a', name='AF'
        ), row=1, col=2)
        fig.add_vline(x=0.5, line_dash='dot', line_color='#457b9d',
                      annotation_text='het', row=1, col=2)
        fig.add_vline(x=1.0, line_dash='dot', line_color='#e63946',
                      annotation_text='hom', row=1, col=2)

        fig.add_trace(go.Histogram(
            x=geno['qual'], nbinsx=60,
            marker_color='#2d6a4f', name='QUAL'
        ), row=2, col=1)

        chrom_order = [str(c) for c in range(1, 23)] + ['X', 'Y', 'MT']
        cc = geno['chrom'].str.replace('chr', '', regex=False).value_counts().reindex(chrom_order).dropna()
        fig.add_trace(go.Bar(
            x=cc.index, y=cc.values, marker_color='#264653', name='Chrom'
        ), row=2, col=2)

        fig.update_layout(
            height=580,
            title=f'{sample_name}  —  {len(geno):,} passing variants',
            showlegend=False
        )
        fig.show()

        median_depth = geno['depth'].median()
        pct_het = ((geno['allele_freq'] >= 0.25) & (geno['allele_freq'] <= 0.75)).mean() * 100
        pct_hom = (geno['allele_freq'] > 0.9).mean() * 100
        print(f'Median depth: {median_depth:.0f}x  |  Het variants: {pct_het:.1f}%  |  Hom variants: {pct_hom:.1f}%')

    def on_wes_change(change):
        with out_wes:
            clear_output(wait=True)
            show_wes_depth(change['new'])

    wes_select.observe(on_wes_change, names='value')
    display(wes_select, out_wes)
    show_wes_depth(wes_runs['run_id'].iloc[0])

## WES vs sc/bulk depth comparison
Side-by-side depth distributions to highlight the difference in coverage between
WES (deep exome) and scRNA/bulk (RNA-seq derived) variant calls.

In [ ]:
if wes_runs.empty or sc_runs.empty:
    print('Need both WES and sc/bulk runs for comparison.')
else:
    # Sample up to 3 runs per type to keep query manageable
    wes_sample = wes_runs['run_id'].head(3).tolist()
    sc_sample  = sc_runs['run_id'].head(3).tolist()

    fig = go.Figure()
    for run_id in wes_sample:
        depths = query("""
            SELECT g.depth FROM genotype_calls g
            WHERE g.run_id=? AND g.filter_status='PASS' LIMIT 100000
        """, (run_id,))
        sname  = wes_runs[wes_runs['run_id'] == run_id]['sample'].iloc[0]
        if not depths.empty:
            fig.add_trace(go.Violin(
                y=np.log10(depths['depth'].clip(lower=1)),
                name=f'WES: {sname}', side='negative',
                line_color='#6d597a', fillcolor='#b5838d',
                box_visible=True, meanline_visible=True, spanmode='hard'
            ))
    for run_id in sc_sample:
        depths = query("""
            SELECT g.depth FROM genotype_calls g
            WHERE g.run_id=? AND g.filter_status='PASS' LIMIT 100000
        """, (run_id,))
        sname  = sc_runs[sc_runs['run_id'] == run_id]['sample'].iloc[0]
        if not depths.empty:
            fig.add_trace(go.Violin(
                y=np.log10(depths['depth'].clip(lower=1)),
                name=f'sc: {sname}', side='positive',
                line_color='#457b9d', fillcolor='#a8dadc',
                box_visible=True, meanline_visible=True, spanmode='hard'
            ))
    fig.update_layout(
        height=480,
        title='WES (purple/left) vs sc/bulk (blue/right) — log10(depth)',
        yaxis=dict(
            title='log10(depth)',
            tickvals=[1, 1.5, 2, 2.5, 3],
            ticktext=['10x', '~30x', '100x', '~300x', '1000x']
        ),
        violingap=0, violingroupgap=0, violinmode='overlay'
    )
    fig.show()

## All-vs-all WES variant overlap
Jaccard similarity and containment matrix for all organoid WES samples.
Samples from the same patient/origin should show higher similarity.

In [ ]:
if len(wes_runs) < 2:
    print('Need at least 2 WES runs for comparison.')
else:
    print('Loading variant keys for WES runs...')
    wes_keys = {}
    for run_id in wes_runs['run_id']:
        df = query("""
            SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS key
            FROM genotype_calls g
            JOIN variants v ON g.variant_id = v.variant_id
            WHERE g.run_id=? AND g.filter_status='PASS'
        """, (run_id,))
        wes_keys[run_id] = set(df['key'].tolist()) if not df.empty else set()

    wes_ids    = wes_runs['run_id'].tolist()
    wes_lbl    = wes_runs['label'].tolist() if 'label' in wes_runs.columns else wes_runs['sample'].tolist()
    n = len(wes_ids)

    jac     = np.zeros((n, n))
    contain = np.zeros((n, n))
    for i, ri in enumerate(wes_ids):
        for j, rj in enumerate(wes_ids):
            si, sj = wes_keys[ri], wes_keys[rj]
            union = len(si | sj)
            inter = len(si & sj)
            jac[i, j]     = inter / union if union else 0
            contain[i, j] = inter / len(si) if si else 0

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Jaccard similarity (symmetric)', 'Containment — row % found in column'),
        horizontal_spacing=0.12
    )
    for mat, col, cscale, cb_x in [
        (jac,     1, 'Purples', 0.44),
        (contain, 2, 'Oranges', 1.01),
    ]:
        fig.add_trace(go.Heatmap(
            z=mat, x=wes_lbl, y=wes_lbl,
            colorscale=cscale, zmin=0, zmax=1,
            text=np.round(mat, 2),
            texttemplate='%{text}',
            hovertemplate='%{y} vs %{x}<br>Value: %{z:.3f}<extra></extra>',
            showscale=True, colorbar=dict(x=cb_x, len=0.9)
        ), row=1, col=col)

    fig.update_layout(
        title=f'WES organoid variant overlap — {n} samples',
        height=max(400, 40 * n),
        width=max(700, 60 * n),
    )
    fig.update_xaxes(tickangle=45)
    fig.show()

    jac_df = pd.DataFrame(jac, index=wes_lbl, columns=wes_lbl)
    upper = jac_df.where(np.triu(np.ones(jac_df.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
    display(upper.reset_index().rename(columns={0: 'jaccard', 'level_0': 'Sample A', 'level_1': 'Sample B'}))

## WES similarity distributions
All-pairs Jaccard and containment distributions for WES samples.
All organoid lines are unrelated donors, so this shows the baseline cross-donor similarity for WES data.

In [ ]:
from scipy.stats import gaussian_kde as _gkde

if len(wes_runs) < 2:
    print('Need at least 2 WES runs for distribution plots.')
else:
    # Reuse wes_keys from the heatmap cell above (must run that cell first)
    wes_ids_list = wes_runs['run_id'].tolist()
    wes_lbl_list = wes_runs['label'].tolist() if 'label' in wes_runs.columns else wes_runs['sample'].tolist()
    m = len(wes_ids_list)

    # Build Jaccard + containment matrices (recompute if not available from heatmap cell)
    try:
        # Try to use wes_keys from the heatmap cell
        _wes_k = wes_keys
    except NameError:
        print('Computing variant keys (run the heatmap cell first for cached keys)...')
        _wes_k = {}
        for run_id in wes_ids_list:
            df = query("""
                SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS key
                FROM genotype_calls g
                JOIN variants v ON g.variant_id = v.variant_id
                WHERE g.run_id=? AND g.filter_status='PASS'
            """, (run_id,))
            _wes_k[run_id] = set(df['key'].tolist()) if not df.empty else set()

    wes_jac  = np.zeros((m, m))
    wes_asym = np.zeros((m, m))
    for i, ri in enumerate(wes_ids_list):
        for j, rj in enumerate(wes_ids_list):
            si, sj = _wes_k[ri], _wes_k[rj]
            union = len(si | sj)
            inter = len(si & sj)
            wes_jac[i, j]  = inter / union if union else 0
            wes_asym[i, j] = inter / len(si) if si else 0

    triu     = np.triu_indices(m, k=1)
    off_diag = ~np.eye(m, dtype=bool)
    jac_vals  = wes_jac[triu]
    asym_vals = wes_asym[off_diag]

    x_grid = np.linspace(0, 1, 300)
    WES_COLOR = '#6d597a'

    # ── Overall distribution (Jaccard + Containment) ──────────────────────────
    fig_dist = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'Jaccard similarity  ({len(jac_vals):,} pairs)',
            f'Containment  ({len(asym_vals):,} values)'
        ),
        horizontal_spacing=0.14
    )
    for col_idx, (vals, x_title) in enumerate(
        [(jac_vals, 'Similarity'), (asym_vals, 'Containment')], start=1
    ):
        ax_num = '' if col_idx == 1 else str(col_idx)
        kde_y  = _gkde(vals)(x_grid) if len(vals) > 1 else np.zeros_like(x_grid)
        med    = float(np.median(vals))
        fig_dist.add_trace(go.Histogram(
            x=vals, histnorm='probability density', nbinsx=40,
            marker_color=WES_COLOR, opacity=0.30, showlegend=False
        ), row=1, col=col_idx)
        fig_dist.add_trace(go.Scatter(
            x=x_grid, y=kde_y, mode='lines',
            line=dict(color=WES_COLOR, width=2.5), showlegend=False
        ), row=1, col=col_idx)
        fig_dist.add_trace(go.Scatter(
            x=[med, med], y=[0, kde_y.max() * 1.15 if kde_y.max() > 0 else 1],
            mode='lines', line=dict(color=WES_COLOR, width=1.5, dash='dot'),
            showlegend=False, hoverinfo='skip'
        ), row=1, col=col_idx)
        fig_dist.add_annotation(
            x=med, y=kde_y.max() * 1.18 if kde_y.max() > 0 else 1.1,
            text=f'median {med:.3f}  n={len(vals)}',
            showarrow=False, font=dict(size=10, color=WES_COLOR),
            xref=f'x{ax_num}', yref=f'y{ax_num}'
        )
        fig_dist.update_xaxes(range=[0, 1], title_text=x_title, row=1, col=col_idx)

    fig_dist.update_yaxes(title_text='Density', row=1, col=1)
    fig_dist.update_layout(height=360, title='WES organoid similarity distributions — all pairs (unrelated donors)')
    fig_dist.show()

    print(f'Jaccard:     median={np.median(jac_vals):.4f}  mean={np.mean(jac_vals):.4f}  n={len(jac_vals)}')
    print(f'Containment: median={np.median(asym_vals):.4f}  mean={np.mean(asym_vals):.4f}  n={len(asym_vals)}')

## WES vs scRNA overlap
For any WES organoid sample that has a matching scRNA run, compare their
shared variant sites — validates whether the organoid WES variants are
captured in the corresponding scRNA experiment.

In [ ]:
if wes_runs.empty or sc_runs.empty:
    print('Need both WES and sc/bulk runs in the DB.')
else:
    wes_samples = set(wes_runs['sample'].str.split('_').str[0])
    sc_samples  = set(sc_runs['sample'].str.split('_').str[0])
    overlap_names = wes_samples & sc_samples
    print(f'Samples present in both WES and sc/bulk: {sorted(overlap_names)}')

    if not overlap_names:
        print('No matching samples between WES and sc/bulk runs.')
    else:
        _key_sql = """
            SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS key
            FROM genotype_calls g
            JOIN variants v ON g.variant_id = v.variant_id
            WHERE g.run_id=? AND g.filter_status='PASS'
        """
        rows = []
        for name in sorted(overlap_names):
            wes_r = wes_runs[wes_runs['sample'].str.startswith(name)]['run_id'].iloc[0]
            sc_r  = sc_runs[sc_runs['sample'].str.startswith(name)]['run_id'].iloc[0]

            wes_k = set(query(_key_sql, (wes_r,))['key'])
            sc_k  = set(query(_key_sql, (sc_r,))['key'])

            inter = len(wes_k & sc_k)
            union = len(wes_k | sc_k)
            rows.append({
                'sample': name,
                'wes_run': wes_r, 'sc_run': sc_r,
                'wes_vars': len(wes_k), 'sc_vars': len(sc_k),
                'shared': inter,
                'jaccard': inter / union if union else 0,
                'wes_recall': inter / len(wes_k) if wes_k else 0,
            })

        comp_df = pd.DataFrame(rows)
        display(comp_df.style.format({
            'wes_vars': '{:,}', 'sc_vars': '{:,}', 'shared': '{:,}',
            'jaccard': '{:.4f}', 'wes_recall': '{:.4f}'
        }))

        fig = go.Figure(data=[
            go.Bar(name='WES variants',     x=comp_df['sample'], y=comp_df['wes_vars'], marker_color='#6d597a'),
            go.Bar(name='sc/bulk variants', x=comp_df['sample'], y=comp_df['sc_vars'],  marker_color='#457b9d'),
            go.Bar(name='Shared variants',  x=comp_df['sample'], y=comp_df['shared'],   marker_color='#2d6a4f'),
        ])
        fig.update_layout(
            barmode='group', height=400,
            title='WES vs sc/bulk variant counts for matched samples',
            yaxis_title='Variant count'
        )
        fig.show()

## Raw SQL explorer

In [ ]:
sql_box = widgets.Textarea(
    value="-- WES runs summary\nSELECT r.run_id, s.name AS sample, r.mapping_rate, r.total_filt, r.run_date\nFROM runs r\nJOIN samples s ON r.sample_id = s.sample_id\nWHERE r.run_id LIKE '%_wes_%'\nORDER BY r.run_date DESC;",
    layout=widgets.Layout(width='100%', height='110px')
)
run_btn = widgets.Button(description='Run query', button_style='primary')

def on_run_query(btn):
    clear_output(wait=False)
    display(sql_box, run_btn)
    try:
        display(query(sql_box.value))
    except Exception as e:
        print(f'Error: {e}')

run_btn.on_click(on_run_query)
display(sql_box, run_btn)